In [1]:
# Compare mean-init and random-init runs directly (CPU only; needs the saved prediction CSVs attached as a dataset)
%pip install -q sacrebleu
import glob, numpy as np, pandas as pd, sacrebleu

FILES = {"baseline v2": "new_baseline_v2_test.csv",
         "mean init, seed 42": "ext_f20_mean_s42_test.csv",
         "mean init, seed 43": "ext_f20_mean_s43_test.csv",
         "random init, seed 42": "ext_f20_random_s42_test.csv"}

def load(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach the predictions as a dataset"
    df = pd.read_csv(hits[0], encoding="utf-8-sig")
    return df["reference"].fillna("").astype(str).str.strip().tolist(), df["prediction"].fillna("").astype(str).str.strip().tolist()

refs, hyps = None, {}
for label, fname in FILES.items():
    r, h = load(fname)
    refs = refs or r
    assert r == refs, f"{fname}: references differ"
    hyps[label] = h
print({k: len(v) for k, v in hyps.items()}, "| references:", len(refs))

rows = [{"system": k, "BLEU": round(sacrebleu.corpus_bleu(v, [refs]).score, 2), "chrF": round(sacrebleu.corpus_chrf(v, [refs]).score, 2),
         "TER": round(sacrebleu.corpus_ter(v, [refs]).score, 2)} for k, v in hyps.items()]
display(pd.DataFrame(rows))

def boot(a, b, B=1000, seed=0):
    rng = np.random.default_rng(seed); n = len(refs); db, dc = [], []
    for _ in range(B):
        idx = rng.integers(0, n, n); r = [refs[i] for i in idx]
        x = [hyps[a][i] for i in idx]; y = [hyps[b][i] for i in idx]
        db.append(sacrebleu.corpus_bleu(y, [r]).score - sacrebleu.corpus_bleu(x, [r]).score)
        dc.append(sacrebleu.corpus_chrf(y, [r]).score - sacrebleu.corpus_chrf(x, [r]).score)
    return db, dc

for a, b in [("random init, seed 42", "mean init, seed 42"), ("random init, seed 42", "mean init, seed 43"),
             ("mean init, seed 42", "mean init, seed 43")]:
    db, dc = boot(a, b)
    for name, d in [("BLEU", db), ("chrF", dc)]:
        lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"{b} minus {a} | {name}: mean {np.mean(d):+.2f}, 95% interval [{lo:+.2f}, {hi:+.2f}]")

Note: you may need to restart the kernel to use updated packages.
{'baseline v2': 994, 'mean init, seed 42': 994, 'mean init, seed 43': 994, 'random init, seed 42': 994} | references: 994


,system,BLEU,chrF,TER
0,baseline v2,12.32,45.26,76.99
1,"mean init, seed 42",13.15,46.15,73.98
2,"mean init, seed 43",13.00,46.17,76.00
3,"random init, seed 42",12.76,46.04,75.11


mean init, seed 42 minus random init, seed 42 | BLEU: mean +0.39, 95% interval [-0.02, +0.80]
mean init, seed 42 minus random init, seed 42 | chrF: mean +0.10, 95% interval [-0.26, +0.47]
mean init, seed 43 minus random init, seed 42 | BLEU: mean +0.24, 95% interval [-0.29, +0.79]
mean init, seed 43 minus random init, seed 42 | chrF: mean +0.13, 95% interval [-0.26, +0.53]


KeyboardInterrupt: 

In [6]:
# Fast paired bootstrap: all runs against BOTH baseline seeds (CPU only; self-checking, falls back to the slow method)
import glob, numpy as np, pandas as pd, sacrebleu

FILES = {"baseline seed 42": "new_baseline_v2_test.csv",
         "baseline seed 43": "baseline_s43_test.csv",
         "mean init 445, seed 42": "ext_f20_mean_s42_test.csv",
         "mean init 445, seed 43": "ext_f20_mean_s43_test.csv",
         "random init 445, seed 42": "ext_f20_random_s42_test.csv"}

def load(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach the predictions as a dataset"
    df = pd.read_csv(hits[0], encoding="utf-8-sig")
    return df["reference"].fillna("").astype(str).str.strip().tolist(), df["prediction"].fillna("").astype(str).str.strip().tolist()

refs, hyps = None, {}
for label, fname in FILES.items():
    r, h = load(fname)
    refs = refs or r
    assert r == refs, f"{fname}: references differ"
    hyps[label] = h

scores = {k: (sacrebleu.corpus_bleu(v, [refs]).score, sacrebleu.corpus_chrf(v, [refs]).score, sacrebleu.corpus_ter(v, [refs]).score)
          for k, v in hyps.items()}
display(pd.DataFrame([{"system": k, "BLEU": round(b, 2), "chrF": round(c, 2), "TER": round(t, 2)} for k, (b, c, t) in scores.items()]))
bs = [scores[k] for k in ("baseline seed 42", "baseline seed 43")]
print("Baseline seed spread: BLEU %.2f, chrF %.2f | baseline mean: BLEU %.2f, chrF %.2f" % (
    abs(bs[0][0] - bs[1][0]), abs(bs[0][1] - bs[1][1]), np.mean([x[0] for x in bs]), np.mean([x[1] for x in bs])))

# --- fast path: per-sentence sufficient statistics, verified against the normal scores ---
FAST, st = True, {}
try:
    from sacrebleu.metrics import BLEU, CHRF
    bleu_m, chrf_m = BLEU(), CHRF()
    for k, h in hyps.items():
        sb_ = bleu_m._extract_corpus_statistics(h, [refs]); sc_ = chrf_m._extract_corpus_statistics(h, [refs])
        assert abs(bleu_m._aggregate_and_compute(sb_).score - sacrebleu.corpus_bleu(h, [refs]).score) < 1e-6
        assert abs(chrf_m._aggregate_and_compute(sc_).score - sacrebleu.corpus_chrf(h, [refs]).score) < 1e-6
        st[k] = (sb_, sc_)
    print("Fast bootstrap path verified.")
except Exception as e:
    FAST = False
    print("Fast path unavailable (" + repr(e)[:100] + "): using the slow method with 200 resamples.")

B = 1000 if FAST else 200
def boot(a, b, seed=0):
    rng = np.random.default_rng(seed); n = len(refs); db, dc = [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if FAST:
            db.append(bleu_m._aggregate_and_compute([st[b][0][i] for i in idx]).score - bleu_m._aggregate_and_compute([st[a][0][i] for i in idx]).score)
            dc.append(chrf_m._aggregate_and_compute([st[b][1][i] for i in idx]).score - chrf_m._aggregate_and_compute([st[a][1][i] for i in idx]).score)
        else:
            r = [refs[i] for i in idx]; x = [hyps[a][i] for i in idx]; y = [hyps[b][i] for i in idx]
            db.append(sacrebleu.corpus_bleu(y, [r]).score - sacrebleu.corpus_bleu(x, [r]).score)
            dc.append(sacrebleu.corpus_chrf(y, [r]).score - sacrebleu.corpus_chrf(x, [r]).score)
    return db, dc

def fmt(d):
    lo, hi = np.percentile(d, [2.5, 97.5]); return f"{np.mean(d):+.2f} [{lo:+.2f}, {hi:+.2f}]"

rows = []
for run in ["mean init 445, seed 42", "mean init 445, seed 43", "random init 445, seed 42"]:
    for base in ["baseline seed 42", "baseline seed 43"]:
        db, dc = boot(base, run)
        rows.append({"run": run, "vs": base, "dBLEU [95% CI]": fmt(db), "dchrF [95% CI]": fmt(dc)})
for a, b in [("baseline seed 42", "baseline seed 43"), ("mean init 445, seed 42", "mean init 445, seed 43"),
             ("random init 445, seed 42", "mean init 445, seed 42"), ("random init 445, seed 42", "mean init 445, seed 43")]:
    db, dc = boot(a, b)
    rows.append({"run": b, "vs": a + ("  (seed spread)" if "seed" in a and a.split(",")[0] == b.split(",")[0] else ""), "dBLEU [95% CI]": fmt(db), "dchrF [95% CI]": fmt(dc)})
display(pd.DataFrame(rows))

,system,BLEU,chrF,TER
0,baseline seed 42,12.32,45.26,76.99
1,baseline seed 43,12.70,45.64,77.07
2,"mean init 445, seed 42",13.15,46.15,73.98
3,"mean init 445, seed 43",13.00,46.17,76.00
4,"random init 445, seed 42",12.76,46.04,75.11


Baseline seed spread: BLEU 0.38, chrF 0.38 | baseline mean: BLEU 12.51, chrF 45.45
Fast bootstrap path verified.


,run,vs,dBLEU [95% CI],dchrF [95% CI]
0,"mean init 445, seed 42",baseline seed 42,"+0.82 [+0.32, +1.41]","+0.89 [+0.44, +1.38]"
1,"mean init 445, seed 42",baseline seed 43,"+0.45 [-0.09, +1.01]","+0.50 [+0.10, +0.92]"
2,"mean init 445, seed 43",baseline seed 42,"+0.67 [+0.01, +1.38]","+0.92 [+0.38, +1.48]"
3,"mean init 445, seed 43",baseline seed 43,"+0.30 [-0.30, +0.89]","+0.53 [+0.05, +0.99]"
4,"random init 445, seed 42",baseline seed 42,"+0.43 [-0.12, +1.02]","+0.78 [+0.31, +1.26]"
5,"random init 445, seed 42",baseline seed 43,"+0.06 [-0.53, +0.65]","+0.40 [-0.04, +0.88]"
6,baseline seed 43,baseline seed 42,"+0.37 [-0.12, +0.88]","+0.38 [-0.02, +0.77]"
7,"mean init 445, seed 43","mean init 445, seed 42 (seed spread)","-0.15 [-0.66, +0.30]","+0.03 [-0.38, +0.39]"
8,"mean init 445, seed 42","random init 445, seed 42","+0.39 [-0.02, +0.80]","+0.10 [-0.26, +0.47]"
9,"mean init 445, seed 43","random init 445, seed 42","+0.24 [-0.29, +0.79]","+0.13 [-0.26, +0.53]"


In [4]:
# Added-word recall/precision vs a control set of equally frequent words that were NOT added (CPU only)
import glob, json, unicodedata, numpy as np, pandas as pd
from collections import Counter

def locate(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach it as a dataset"
    return hits[0]

FILES = {"baseline seed 42": "new_baseline_v2_test.csv", "baseline seed 43": "baseline_s43_test.csv",
         "mean init, seed 42": "ext_f20_mean_s42_test.csv", "mean init, seed 43": "ext_f20_mean_s43_test.csv",
         "random init, seed 42": "ext_f20_random_s42_test.csv"}
refs, hyps = None, {}
for label, fname in FILES.items():
    df = pd.read_csv(locate(fname), encoding="utf-8-sig")
    r = df["reference"].fillna("").astype(str).str.strip().tolist()
    refs = refs or r
    assert r == refs
    hyps[label] = df["prediction"].fillna("").astype(str).str.strip().tolist()

added = set(json.load(open(locate("candidate_lists_by_threshold.json"), encoding="utf-8"))["20"])
train = pd.read_parquet(locate("santali_train.parquet"))["Santali"].dropna().astype(str)
train_counts = Counter(w for t in train for w in t.split())
control = {w for w, f in train_counts.items() if f >= 20 and w not in added}
print("Added words:", len(added), "| control words (frequent, not added):", len(control))

def strip_edge(t):
    s, e = 0, len(t)
    while s < e and unicodedata.category(t[s])[0] in {"P", "S"}: s += 1
    while e > s and unicodedata.category(t[e - 1])[0] in {"P", "S"}: e -= 1
    return t[s:e]

def per_sentence(hyp, vocab):
    ref_n, hyp_n, match = [], [], []
    for r, h in zip(refs, hyp):
        rc = Counter(strip_edge(t) for t in r.split()); rc = Counter({w: c for w, c in rc.items() if w in vocab})
        hc = Counter(strip_edge(t) for t in h.split()); hc = Counter({w: c for w, c in hc.items() if w in vocab})
        ref_n.append(sum(rc.values())); hyp_n.append(sum(hc.values())); match.append(sum(min(c, hc[w]) for w, c in rc.items()))
    return np.array(ref_n), np.array(hyp_n), np.array(match)

data = {(s, name): per_sentence(h, v) for s, h in hyps.items() for name, v in [("added", added), ("control", control)]}
rows = []
for s in hyps:
    row = {"system": s}
    for name in ("added", "control"):
        rn, hn, m = data[(s, name)]
        row[f"{name} recall %"] = round(100 * m.sum() / rn.sum(), 2)
        row[f"{name} precision %"] = round(100 * m.sum() / max(hn.sum(), 1), 2)
    rows.append(row)
display(pd.DataFrame(rows))
print("Reference occurrences: added", int(data[("baseline seed 42", "added")][0].sum()), "| control", int(data[("baseline seed 42", "control")][0].sum()))

rng = np.random.default_rng(0); n = len(refs)
idxs = [rng.integers(0, n, n) for _ in range(2000)]
def rec(s, name, idx):
    rn, _, m = data[(s, name)]; return m[idx].sum() / rn[idx].sum()
out = []
for run in ["mean init, seed 42", "mean init, seed 43", "random init, seed 42"]:
    for name in ("added", "control"):
        d = [100 * (rec(run, name, i) - 0.5 * (rec("baseline seed 42", name, i) + rec("baseline seed 43", name, i))) for i in idxs]
        lo, hi = np.percentile(d, [2.5, 97.5])
        out.append({"run": run, "word set": name, "recall change vs baseline average (points)": f"{np.mean(d):+.2f} [{lo:+.2f}, {hi:+.2f}]"})
display(pd.DataFrame(out))

Added words: 445 | control words (frequent, not added): 1328


,system,added recall %,added precision %,control recall %,control precision %
0,baseline seed 42,41.43,26.56,51.85,44.25
1,baseline seed 43,41.70,25.46,52.39,44.79
2,"mean init, seed 42",43.45,29.57,53.24,45.90
3,"mean init, seed 43",42.80,30.48,53.41,44.55
4,"random init, seed 42",44.36,31.76,53.23,45.11


Reference occurrences: added 1091 | control 12759


,run,word set,recall change vs baseline average (points)
0,"mean init, seed 42",added,"+1.88 [-0.22, +4.04]"
1,"mean init, seed 42",control,"+1.12 [+0.65, +1.59]"
2,"mean init, seed 43",added,"+1.26 [-0.74, +3.33]"
3,"mean init, seed 43",control,"+1.30 [+0.79, +1.84]"
4,"random init, seed 42",added,"+2.81 [+0.79, +4.85]"
5,"random init, seed 42",control,"+1.11 [+0.62, +1.59]"


In [5]:
# Added-word recall/precision vs a control set of equally frequent words that were NOT added (CPU only)
import glob, json, unicodedata, numpy as np, pandas as pd
from collections import Counter

def locate(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach it as a dataset"
    return hits[0]

FILES = {"baseline seed 42": "new_baseline_v2_test.csv", "baseline seed 43": "baseline_s43_test.csv",
         "mean init, seed 42": "ext_f20_mean_s42_test.csv", "mean init, seed 43": "ext_f20_mean_s43_test.csv",
         "random init, seed 42": "ext_f20_random_s42_test.csv"}
refs, hyps = None, {}
for label, fname in FILES.items():
    df = pd.read_csv(locate(fname), encoding="utf-8-sig")
    r = df["reference"].fillna("").astype(str).str.strip().tolist()
    refs = refs or r
    assert r == refs
    hyps[label] = df["prediction"].fillna("").astype(str).str.strip().tolist()

lists = json.load(open(locate("candidate_lists_by_threshold.json"), encoding="utf-8"))
added = set(lists["20"])
control = set(lists["10"]) - set(lists["20"])   # multi-piece words with frequency 10-19: same tokenization difficulty, NOT added in the 445-token runs
train = pd.read_parquet(locate("santali_train.parquet"))["Santali"].dropna().astype(str)
train_counts = Counter(w for t in train for w in t.split())

print("Added words:", len(added), "| control words (frequent, not added):", len(control))

def strip_edge(t):
    s, e = 0, len(t)
    while s < e and unicodedata.category(t[s])[0] in {"P", "S"}: s += 1
    while e > s and unicodedata.category(t[e - 1])[0] in {"P", "S"}: e -= 1
    return t[s:e]

def per_sentence(hyp, vocab):
    ref_n, hyp_n, match = [], [], []
    for r, h in zip(refs, hyp):
        rc = Counter(strip_edge(t) for t in r.split()); rc = Counter({w: c for w, c in rc.items() if w in vocab})
        hc = Counter(strip_edge(t) for t in h.split()); hc = Counter({w: c for w, c in hc.items() if w in vocab})
        ref_n.append(sum(rc.values())); hyp_n.append(sum(hc.values())); match.append(sum(min(c, hc[w]) for w, c in rc.items()))
    return np.array(ref_n), np.array(hyp_n), np.array(match)

data = {(s, name): per_sentence(h, v) for s, h in hyps.items() for name, v in [("added", added), ("control", control)]}
rows = []
for s in hyps:
    row = {"system": s}
    for name in ("added", "control"):
        rn, hn, m = data[(s, name)]
        row[f"{name} recall %"] = round(100 * m.sum() / rn.sum(), 2)
        row[f"{name} precision %"] = round(100 * m.sum() / max(hn.sum(), 1), 2)
    rows.append(row)
display(pd.DataFrame(rows))
print("Reference occurrences: added", int(data[("baseline seed 42", "added")][0].sum()), "| control", int(data[("baseline seed 42", "control")][0].sum()))

rng = np.random.default_rng(0); n = len(refs)
idxs = [rng.integers(0, n, n) for _ in range(2000)]
def rec(s, name, idx):
    rn, _, m = data[(s, name)]; return m[idx].sum() / rn[idx].sum()
out = []
for run in ["mean init, seed 42", "mean init, seed 43", "random init, seed 42"]:
    for name in ("added", "control"):
        d = [100 * (rec(run, name, i) - 0.5 * (rec("baseline seed 42", name, i) + rec("baseline seed 43", name, i))) for i in idxs]
        lo, hi = np.percentile(d, [2.5, 97.5])
        out.append({"run": run, "word set": name, "recall change vs baseline average (points)": f"{np.mean(d):+.2f} [{lo:+.2f}, {hi:+.2f}]"})
display(pd.DataFrame(out))

Added words: 445 | control words (frequent, not added): 775


,system,added recall %,added precision %,control recall %,control precision %
0,baseline seed 42,41.43,26.56,32.26,31.45
1,baseline seed 43,41.70,25.46,33.27,32.17
2,"mean init, seed 42",43.45,29.57,32.67,31.90
3,"mean init, seed 43",42.80,30.48,32.46,32.46
4,"random init, seed 42",44.36,31.76,32.06,31.56


Reference occurrences: added 1091 | control 499


,run,word set,recall change vs baseline average (points)
0,"mean init, seed 42",added,"+1.88 [-0.22, +4.04]"
1,"mean init, seed 42",control,"-0.08 [-1.65, +1.45]"
2,"mean init, seed 43",added,"+1.26 [-0.74, +3.33]"
3,"mean init, seed 43",control,"-0.27 [-2.21, +1.45]"
4,"random init, seed 42",added,"+2.81 [+0.79, +4.85]"
5,"random init, seed 42",control,"-0.69 [-2.30, +0.81]"
